# Permian Basin RRC Production Comparison

**Counties:** Midland, Martin, Howard, Reeves, Loving  
**Period:** January 2020 - March 2026  
**Source:** Texas Railroad Commission Production Data Query (PDQ)

This notebook reproduces the Part 2 analysis using the companion `Permian_Basin_Production_Analysis.py` module. The core business-focused metrics are production scale and growth, total-liquids mix, gas intensity, operator/field concentration, and Q1 2026 momentum.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from Permian_Basin_Production_Analysis import (
    COUNTIES, load_monthly, load_ranked_report, build_annual,
    build_metrics, top_n_with_share, save_figures
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load and standardize RRC exports

In [2]:
monthly = pd.concat([load_monthly(c) for c in COUNTIES], ignore_index=True)
annual = build_annual(monthly)
operators = pd.concat([load_ranked_report(c, "Operator") for c in COUNTIES], ignore_index=True)
fields = pd.concat([load_ranked_report(c, "Field") for c in COUNTIES], ignore_index=True)

print(f"Monthly rows: {len(monthly):,}")
print(f"Annual county-year rows: {len(annual):,}")
display(monthly.head())

Monthly rows: 375
Annual county-year rows: 35


,Date,Oil (BBL),Casinghead (MCF),GW Gas (MCF),Condensate (BBL),Gas (MCF),County,Year,Liquids (BBL),GOR (MCF/BBL Oil),GLR (MCF/BBL Liquids)
0,2020-01-01,17030249,41328706,321266,6888,41649972,Midland,2020,17037137,2.45,2.44
1,2020-02-01,15772319,38415798,312327,6636,38728125,Midland,2020,15778955,2.46,2.45
2,2020-03-01,17266683,44144100,370431,7230,44514531,Midland,2020,17273913,2.58,2.58
3,2020-04-01,15963725,43160681,339924,6056,43500605,Midland,2020,15969781,2.72,2.72
4,2020-05-01,14639625,40822467,337548,5265,41160015,Midland,2020,14644890,2.81,2.81


## 2. Build the executive county comparison

In [3]:
metrics = build_metrics(monthly, annual, operators, fields)
display(metrics.round(2))

,County,2025 Oil (MMbbl),2020-2025 Oil Growth (%),2020-2025 Oil CAGR (%),2025 Gas (Bcf),2020-2025 Gas Growth (%),2020-2025 Gas CAGR (%),2025 Condensate (MMbbl),2025 Total Liquids (MMbbl),2025 Condensate Share of Liquids (%),2020 GLR (MCF/BBL Liquids),2025 GLR (MCF/BBL Liquids),2020-2025 GLR Change (%),2026 Q1 Oil Momentum vs 2025 Avg (%),2026 Q1 Gas Momentum vs 2025 Avg (%),Top Oil Operator (2020-Mar 2026),Top Operator Oil Share (%),Top 5 Operator Oil Share (%),Top Oil Field (2020-Mar 2026),Top Field Oil Share (%)
0,Midland,241.69,26.09,4.75,982.60,78.72,12.31,0.76,242.46,0.32,2.87,4.05,41.34,-7.32,-5.30,"PIONEER NATURAL RES. USA, INC.",33.47,71.34,SPRABERRY (TREND AREA),81.14
1,Martin,258.33,76.78,12.07,749.73,155.15,20.60,0.42,258.74,0.16,2.01,2.90,44.09,-0.92,3.24,DIAMONDBACK E&P LLC,28.59,71.65,SPRABERRY (TREND AREA),96.72
2,Howard,98.59,1.73,0.34,426.00,127.31,17.85,0.00,98.60,0.00,1.93,4.32,123.44,-21.01,-3.53,SM ENERGY COMPANY,14.15,51.76,SPRABERRY (TREND AREA),97.23
3,Reeves,81.98,-10.61,-2.22,"1,218.87",14.38,2.72,97.87,179.85,54.42,5.66,6.78,19.79,-9.77,-6.21,NOBLE ENERGY INC,13.27,47.64,WOLFBONE (TREND AREA),39.79
4,Loving,113.75,36.90,6.48,702.04,67.21,10.83,63.77,177.52,35.92,3.11,3.95,27.14,-1.82,-1.71,ANADARKO E&P ONSHORE LLC,37.14,77.86,PHANTOM (WOLFCAMP),83.71


### Why total liquids matter
Oil alone is a poor cross-basin liquids comparison because condensate is material in Reeves and Loving. The analysis therefore tracks both **Oil** and **Total Liquids = Oil + Condensate**, and uses **GLR = Gas / Total Liquids** for product-mix comparisons.

In [4]:
mix_cols = [
    "County", "2025 Oil (MMbbl)", "2025 Condensate (MMbbl)",
    "2025 Total Liquids (MMbbl)", "2025 Condensate Share of Liquids (%)",
    "2025 GLR (MCF/BBL Liquids)"
]
display(metrics[mix_cols].round(2))

,County,2025 Oil (MMbbl),2025 Condensate (MMbbl),2025 Total Liquids (MMbbl),2025 Condensate Share of Liquids (%),2025 GLR (MCF/BBL Liquids)
0,Midland,241.69,0.76,242.46,0.32,4.05
1,Martin,258.33,0.42,258.74,0.16,2.90
2,Howard,98.59,0.00,98.60,0.00,4.32
3,Reeves,81.98,97.87,179.85,54.42,6.78
4,Loving,113.75,63.77,177.52,35.92,3.95


## 3. Operator and field concentration

In [5]:
top_operators = top_n_with_share(operators, "Operator Name", 10)
top_fields = top_n_with_share(fields, "Field Name", 5)

display(top_operators.groupby("County").head(5).round(2))
display(top_fields.groupby("County").head(3).round(2))

,County,Rank,Operator Name,Oil (BBL),Gas (MCF),Condensate (BBL),Oil Share (%)
0,Midland,1,"PIONEER NATURAL RES. USA, INC.",461756167,1573697233,63571,33.47
1,Midland,2,ENDEAVOR ENERGY RESOURCES L.P.,151813457,437944654,35429,11.01
2,Midland,3,XTO ENERGY INC.,144255030,698852985,68304,10.46
3,Midland,4,CHEVRON U. S. A. INC.,114741932,359035478,5096,8.32
4,Midland,5,DIAMONDBACK E&P LLC,111512115,469461067,4366,8.08
10,Martin,1,DIAMONDBACK E&P LLC,370960754,922098212,0,28.59
11,Martin,2,"PIONEER NATURAL RES. USA, INC.",252522475,575879247,0,19.47
12,Martin,3,ENDEAVOR ENERGY RESOURCES L.P.,122061636,294678846,301,9.41
13,Martin,4,OVINTIV USA INC.,93679011,263253632,0,7.22
14,Martin,5,XTO ENERGY INC.,90302576,311822207,0,6.96


,County,Rank,Field Name,Oil (BBL),Gas (MCF),Condensate (BBL),Oil Share (%)
0,Midland,1,SPRABERRY (TREND AREA),1119241146,3628915534,8176,81.14
1,Midland,2,SPRABERRY (TREND AREA) R 40 EXC,129676142,787274092,2046,9.40
2,Midland,3,PARKS (CONSOLIDATED),108221412,294759617,70476,7.85
5,Martin,1,SPRABERRY (TREND AREA),1254792046,3099574753,0,96.72
6,Martin,2,SPRABERRY (TREND AREA) R 40 EXC,26109286,63096584,0,2.01
7,Martin,3,EMMA (BARNETT SHALE),12070196,55546182,1262789,0.93
10,Howard,1,SPRABERRY (TREND AREA),692121345,1994038708,4050,97.23
11,Howard,2,HOWARD GLASSCOCK (CONSOLIDATED),8418503,750308,0,1.18
12,Howard,3,SPRABERRY (TREND AREA) R 40 EXC,5520063,10875046,0,0.78
15,Reeves,1,WOLFBONE (TREND AREA),202414450,540845340,817623,39.79


## 4. Generate report figures

In [6]:
save_figures(annual, metrics)
print("Figures refreshed in ../Figures/")

Figures refreshed in ../Figures/


## Key takeaways

- **Martin** is the strongest growth county in the set, with 2020-2025 oil growth of about 77% and gas growth of about 155%.
- **Reeves** is the largest 2025 gas producer, but condensate is more than half of its liquids production, so oil-only comparisons understate its liquids scale.
- **Howard** shows the largest change in gas intensity: oil is nearly flat versus 2020 while gas more than doubled.
- **Loving** has the highest top-five reported operator concentration; **Reeves** has the lowest.
- Q1 2026 oil run rates are below 2025 averages in all five counties, with Martin and Loving holding up best and Howard weakening the most.

> **Operator-name caveat:** RRC operator reports aggregate the full query period, so historical operator names can remain after acquisitions or asset transfers. This is a production-concentration screen, not a current acreage-ownership map.